# Setup: EvalHub SDK Configuration

This notebook configures the [eval-hub-sdk](https://github.com/eval-hub/eval-hub-sdk) to run LLM evaluations against an **already-deployed EvalHub service** on OpenShift AI, with **MLflow experiment tracking**.

## What is EvalHub?

EvalHub is a lightweight REST API service that orchestrates LLM evaluations across multiple backends. It:

- Routes evaluation requests to frameworks like **lm-evaluation-harness**, RAGAS, GuideLLM, LightEval, and more
- Tracks experiments via **MLflow** (metrics, parameters, artifacts)
- Runs natively on **OpenShift** via the TrustyAI Operator
- Supports a **"Bring Your Own Framework" (BYOF)** approach through the SDK

## EvalHub Features (GA in RHOAI 3.5)

| Feature | Description |
|---------|-------------|
| Interface | REST API + Python SDK |
| Frameworks | lm-evaluation-harness, GuideLLM, RAGAS, LightEval, and more |
| Experiment tracking | Built-in MLflow integration |
| Multi-benchmark jobs | Multiple benchmarks per request |
| Result management | Centralized API + MLflow UI |

## Prerequisites

- EvalHub and MLflow already deployed on the cluster
- `.env` file configured with `EVALHUB_URL`, `MODEL_ENDPOINT`, `MODEL_API_KEY` (see `sample.env`)

### Step 0: Load Configuration

Configuration is loaded from `../.env`. Copy `sample.env` to `.env` and update values before running.

| Scenario | What to set |
|----------|-------------|
| **Cluster owner** (deploying EvalHub yourself) | `NAMESPACE`, `MODEL_NAME`, `MODEL_ENDPOINT`, `MODEL_API_KEY` |
| **Workshop participant** (using someone else's cluster) | All values from cluster owner: `NAMESPACE`, `MODEL_NAME`, `MODEL_ENDPOINT`, `MODEL_API_KEY`, `EVALHUB_URL`, `EVALHUB_AUTH_TOKEN`, `MLFLOW_TRACKING_URI` |

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(dotenv_path="../.env", override=True)

NAMESPACE         = os.getenv("NAMESPACE", "demo")
MODEL_NAME        = os.getenv("MODEL_NAME", "glm-53-flash")
MODEL_ENDPOINT    = os.getenv("MODEL_ENDPOINT", "")
MODEL_API_KEY     = os.getenv("MODEL_API_KEY", "")
LIMIT             = os.getenv("LIMIT", "5")
BATCH_SIZE        = os.getenv("BATCH_SIZE", "8")
EVALHUB_URL       = os.getenv("EVALHUB_URL", "")
import subprocess
EVALHUB_AUTH_TOKEN = os.getenv("EVALHUB_AUTH_TOKEN", "")
if not EVALHUB_AUTH_TOKEN:
    _r = subprocess.run(["oc", "whoami", "-t"], capture_output=True, text=True)
    if _r.returncode == 0:
        EVALHUB_AUTH_TOKEN = _r.stdout.strip()
        print("Auth: using oc token")
MLFLOW_TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI", "https://mlflow.redhat-ods-applications.svc:8443")

print(f"Namespace:       {NAMESPACE}")
print(f"Model Name:      {MODEL_NAME}")
print(f"Model Endpoint:  {MODEL_ENDPOINT}")
print(f"Model API Key:   {'***' + MODEL_API_KEY[-4:] if MODEL_API_KEY else 'Not set'}")
print(f"EvalHub URL:     {EVALHUB_URL or '(auto-detect)'}")
print(f"MLflow URI:      {MLFLOW_TRACKING_URI}")

In [ ]:
%pip install -q -r ../requirements.txt

### Step 1: Resolve Authentication Token

EvalHub API calls require a **Bearer token**. This is loaded from `EVALHUB_AUTH_TOKEN` in `.env`, or falls back to `oc whoami -t`.

In [ ]:
import subprocess

if EVALHUB_AUTH_TOKEN:
    print(f"Using token from .env: {EVALHUB_AUTH_TOKEN[:10]}...")
else:
    result = subprocess.run(["oc", "whoami", "-t"], capture_output=True, text=True)
    if result.returncode == 0:
        EVALHUB_AUTH_TOKEN = result.stdout.strip()
        print(f"Token from oc whoami: {EVALHUB_AUTH_TOKEN[:10]}...")
    else:
        print("No token available. Set EVALHUB_AUTH_TOKEN in .env or run: oc login <cluster-url>")
        EVALHUB_AUTH_TOKEN = None

### Step 2: Verify EvalHub Connectivity

Check that EvalHub is reachable at the URL from `.env`:

In [ ]:
import httpx

try:
    resp = httpx.get(f"{EVALHUB_URL}/api/v1/health", verify=False, timeout=5)
    info = resp.json()
    print(f"[OK] EvalHub reachable at {EVALHUB_URL}")
    print(f"     status={info.get('status')}, version={info.get('build', '?')}")
except Exception as e:
    print(f"[FAIL] EvalHub not reachable at {EVALHUB_URL}")
    print(f"       Error: {e}")
    print(f"       Check EVALHUB_URL in .env")

---

## Part B: Configure the EvalHub SDK

### Step 3: Initialize the SDK Client

In [ ]:
from evalhub import SyncEvalHubClient

client = SyncEvalHubClient(
    base_url=EVALHUB_URL,
    auth_token=EVALHUB_AUTH_TOKEN,
    insecure=True,
    tenant=NAMESPACE,
)

print(f"EvalHub client initialized: {EVALHUB_URL}")
print(f"Tenant (namespace):         {NAMESPACE}")

### Step 4: Explore Available Providers and Benchmarks

EvalHub ships with pre-configured providers. Let's list them and their benchmarks.

In [ ]:
try:
    providers = client.providers.list()
    print(f"Available Providers ({len(providers)}):")
    print("=" * 60)
    for provider in providers:
        print(f"\n  Provider: {provider.name}")
        print(f"  ID:       {provider.resource.id}")
        print(f"  Desc:     {provider.description}")
        print(f"  Benchmarks: {len(provider.benchmarks)}")
except Exception as e:
    print(f"Failed to list providers: {e}")

In [ ]:
try:
    benchmarks = client.benchmarks.list()
    print(f"\nAvailable Benchmarks ({len(benchmarks)}):")
    print("=" * 60)
    for bm in benchmarks[:20]:
        print(f"  {bm.id:30s}  category={bm.category or 'N/A':15s}  metrics={bm.metrics}")
    if len(benchmarks) > 20:
        print(f"  ... and {len(benchmarks) - 20} more")
except Exception as e:
    print(f"Failed to list benchmarks: {e}")
    benchmarks = None

### Register Korean MCQ Benchmarks

The default EvalHub catalog does not include Korean-language MCQ benchmarks with **per-question accuracy tracking**.
We register a **custom adapter provider** (`korean_mcq`) that uses a dedicated container image
to evaluate Korean LLMs on CLIcK, HAE-RAE, KMMLU, and KMMLU-HARD benchmarks.

Unlike the built-in `lm-evaluation-harness` provider, this adapter:
- Calls the vLLM API directly with MCQ-formatted prompts
- **Async parallel processing** (configurable concurrency, default 20) for ~10x faster evaluation
- Records per-question answers (correct/incorrect) in CSV
- Reports category-level and supercategory-level accuracy to MLflow
- Generates a detailed markdown report as an OCI artifact
- Robust error handling with retries for rate limits, timeouts, and connection errors

The provider definition lives in [`adapters/korean-mcq/provider.yaml`](../adapters/korean-mcq/provider.yaml)
following the [eval-hub-contrib](https://github.com/eval-hub/eval-hub-contrib) `provider.yaml` format.
See [`adapters/korean-mcq/README.md`](../adapters/korean-mcq/README.md) for full documentation.

#### Build the Korean MCQ Adapter Image

The adapter runs as a container in the cluster. We build the image using OpenShift's internal registry
so that EvalHub can pull it. The image is built in the current `NAMESPACE`.

> **Note:** This only needs to run once per namespace. If the image already exists, the cell will skip the build.

In [ ]:
import subprocess

ADAPTER_DIR = "../adapters/korean-mcq"
IMAGE_NAME = "korean-mcq-adapter"

def _image_exists(namespace: str, name: str) -> bool:
    r = subprocess.run(
        ["oc", "get", "imagestream", name, "-n", namespace],
        capture_output=True, text=True,
    )
    return r.returncode == 0

if _image_exists(NAMESPACE, IMAGE_NAME):
    print(f"Image '{IMAGE_NAME}' already exists in '{NAMESPACE}' — skipping build.")
else:
    print(f"Building '{IMAGE_NAME}' in namespace '{NAMESPACE}'...")
    print("This takes 1-2 minutes on first run.\n")

    r = subprocess.run(
        ["oc", "new-build", "--strategy=docker", "--binary",
         f"--name={IMAGE_NAME}", "-n", NAMESPACE],
        capture_output=True, text=True,
    )
    if r.returncode != 0 and "already exists" not in r.stderr:
        print(f"new-build failed: {r.stderr}")
    else:
        print("BuildConfig created.")

    r = subprocess.run(
        ["oc", "start-build", IMAGE_NAME,
         f"--from-dir={ADAPTER_DIR}", "--follow", "-n", NAMESPACE],
        text=True,
    )
    if r.returncode == 0:
        print(f"\nImage built successfully: {IMAGE_NAME}:latest in {NAMESPACE}")
    else:
        print(f"\nBuild failed (exit={r.returncode}). Check: oc logs bc/{IMAGE_NAME} -n {NAMESPACE}")

In [ ]:
import yaml, json, pathlib, httpx

KOREAN_PROVIDER_YAML = pathlib.Path("../adapters/korean-mcq/provider.yaml")
KOREAN_PROVIDER_NAME = "Korean MCQ Evaluation"

def _register_korean_provider(client, yaml_path: pathlib.Path) -> str | None:
    """Register the Korean MCQ adapter provider via the REST API (idempotent).
    Returns the provider_id (UUID assigned by EvalHub) or None on failure."""
    for p in client.providers.list():
        if p.name == KOREAN_PROVIDER_NAME:
            print(f"Provider '{KOREAN_PROVIDER_NAME}' already registered (id={p.resource.id}).")
            return p.resource.id

    raw = yaml_path.read_text().replace("${NAMESPACE}", NAMESPACE)
    provider_def = yaml.safe_load(raw)

    resp = httpx.post(
        f"{EVALHUB_URL}/api/v1/evaluations/providers",
        headers={
            "Authorization": f"Bearer {EVALHUB_AUTH_TOKEN}",
            "Content-Type": "application/json",
            "X-Tenant": NAMESPACE,
        },
        json=provider_def,
        verify=False,
        timeout=10,
    )
    if resp.status_code in (200, 201):
        data = resp.json()
        pid = data["resource"]["id"]
        n = len(data.get("benchmarks") or [])
        print(f"Provider '{provider_def['name']}' registered (id={pid}, {n} benchmarks)")
        return pid
    else:
        print(f"Registration failed ({resp.status_code}): {resp.text[:200]}")
        return None

KOREAN_PROVIDER_ID = _register_korean_provider(client, KOREAN_PROVIDER_YAML)

benchmarks = client.benchmarks.list()
korean_keywords = ["kmmlu", "haerae", "click", "korean_mcq"]
korean_benchmarks = [
    bm for bm in benchmarks
    if any(kw in bm.id.lower() for kw in korean_keywords)
]
print(f"\nKorean MCQ Benchmarks Found: {len(korean_benchmarks)}")
print("=" * 60)
for bm in korean_benchmarks:
    print(f"  {bm.id:35s}  {bm.name}")

### Step 5: Configure the Model Endpoint

The `ModelConfig` specifies which MaaS model endpoint EvalHub should target.

#### Key Parameters

| Parameter | Description | Example |
|-----------|-------------|---------|
| `url` | OpenAI-compatible endpoint URL | `https://maas.example.com/model` |
| `name` | Model name | `glm-53-flash` |

In [ ]:
import sys; sys.path.insert(0, '..')
from utils.model_auth import build_model_config

model = build_model_config(MODEL_ENDPOINT, MODEL_NAME, MODEL_API_KEY, NAMESPACE)

print("Model Configuration:")
print(f"  URL:   {model.url}")
print(f"  Name:  {model.name}")
print(f"  Auth:  {model.auth.secret_ref if model.auth else 'None'}")

### Step 6: Configure MLflow Experiment Tracking

EvalHub integrates with MLflow to automatically track evaluation metrics, parameters, and artifacts. When you include an `ExperimentConfig` in your job submission, EvalHub will:

1. Create (or reuse) an MLflow experiment with the given name
2. Log all benchmark metrics (accuracy, f1, etc.) as MLflow metrics
3. Tag the run with model info, benchmark details, and custom tags
4. Store detailed result artifacts

The MLflow connection was configured in **Step A-4** via `MLFLOW_TRACKING_URI` in the EvalHub CR.

#### What MLflow Records

| MLflow Tab | Recorded? | Description |
|------------|-----------|-------------|
| **Overview** | Yes | Job metadata, parameters (model, benchmark, temperature, concurrency, etc.) |
| **Model Metrics** | Yes | `overall_accuracy`, `category_accuracy.*`, `supercategory_accuracy.*` |
| **Artifacts** | Yes | `detailed_results.csv`, `results.json`, `DETAILED_RESULTS.md` |
| **Trace** | Yes | Each LLM call (prompt/response) is recorded as a structured trace span via `MlflowClient.start_trace()` API, visible in the MLflow UI's Traces tab. The adapter connects directly to the MLflow service (bypassing EvalHub proxy). |
| **System Metrics** | No | Requires `psutil` + `mlflow.enable_system_metrics_logging()` in the Pod |

> **Note:** System Metrics are not recorded by design. The Korean MCQ adapter prioritizes
> lightweight, fast execution within a K8s Pod.

#### ExperimentConfig in Job Submission

You control experiment tracking per-job via the `experiment` field:

In [ ]:
from evalhub import ExperimentConfig, ExperimentTag

experiment = ExperimentConfig(
    name="korean-llm-evaluation",
    tags=[
        ExperimentTag(key="model", value=MODEL_NAME),
        ExperimentTag(key="language", value="korean"),
        ExperimentTag(key="environment", value="dev"),
        ExperimentTag(key="team", value="ai-evaluation"),
    ],
)

print("MLflow Experiment Configuration:")
print(f"  Name:  {experiment.name}")
print(f"  Tags:")
for tag in experiment.tags:
    print(f"    {tag.key}: {tag.value}")

### Step 7: Submit a Single Benchmark Evaluation

Submit a **CLIcK** (Cultural and Linguistic Intelligence in Korean) benchmark evaluation
using the `korean_mcq` provider. This is a quick smoke test with 20 questions.
For a full evaluation, use **3_eval_hub_unified_benchmark/**.

In [ ]:
from evalhub import BenchmarkConfig, JobSubmissionRequest

single_job_request = JobSubmissionRequest(
    name="click-evaluation",
    description="CLIcK Korean cultural/linguistic MCQ benchmark",
    tags=["korean", "click", "mcq", "culture"],
    model=model,
    benchmarks=[
        BenchmarkConfig(
            id="click",
            provider_id=KOREAN_PROVIDER_ID,
            parameters={
                "temperature": 0.0,
                "max_tokens": 1024,
                "limit": 20,
            },
        ),
    ],
    experiment=experiment,
)

print("Job Submission Request:")
print(f"  Name:       {single_job_request.name}")
print(f"  Model:      {single_job_request.model.name} @ {single_job_request.model.url}")
print(f"  Provider:   {KOREAN_PROVIDER_ID}")
print(f"  Benchmarks: {[b.id for b in single_job_request.benchmarks]}")
print(f"  Experiment: {single_job_request.experiment.name}")

In [ ]:
try:
    job = client.jobs.submit(single_job_request)
    print(f"Job submitted!")
    print(f"  Job ID:        {job.id}")
    print(f"  State:         {job.state}")
    print(f"  MLflow Exp ID: {job.resource.mlflow_experiment_id or 'pending'}")
except Exception as e:
    job = None
    print(f"Failed to submit job: {e}")

### Step 8: Monitor Job Progress

Poll the job status until it completes.

In [ ]:
import time
from evalhub import JobStatus

if job is None:
    print("Skipped — no job was submitted in the previous cell.")
else:
    TERMINAL_STATES = {JobStatus.COMPLETED, JobStatus.FAILED, JobStatus.CANCELLED, JobStatus.PARTIALLY_FAILED}

    print(f"Monitoring job {job.id}...")
    print("-" * 60)

    while True:
        status = client.jobs.get(job.id)
        state = status.effective_state

        msg = ""
        if status.status and status.status.message:
            msg = f" - {status.status.message.message}"
        print(f"  [{state.value:>10s}]{msg}")

        if state in TERMINAL_STATES:
            break

        time.sleep(10)

    print("-" * 60)
    print(f"Final state: {state.value}")

### Step 9: View Results

Retrieve the evaluation results, including MLflow run information.

In [ ]:
if job is None:
    print("Skipped — no job was submitted.")
else:
    completed_job = client.jobs.get(job.id)

    if completed_job.results:
        print("Evaluation Results:")
        print("=" * 60)

        if completed_job.results.mlflow_experiment_url:
            print(f"\n  MLflow Experiment: {completed_job.results.mlflow_experiment_url}")

        for bm_result in completed_job.results.benchmarks:
            print(f"\n  Benchmark: {bm_result.id} (provider: {bm_result.provider_id})")
            if bm_result.mlflow_run_id:
                print(f"  MLflow Run ID: {bm_result.mlflow_run_id}")
            print(f"  Metrics:")
            for metric_name, metric_value in bm_result.metrics.items():
                print(f"    {metric_name}: {metric_value}")
    else:
        print("No results available yet.")

### Step 10: Multi-Benchmark Evaluation

Submit multiple benchmarks in a single request. EvalHub runs them concurrently and tracks all results under one MLflow experiment.

In [ ]:
multi_job_request = JobSubmissionRequest(
    name="korean-mcq-comprehensive-eval",
    description="Multi-benchmark Korean MCQ evaluation (CLIcK + KMMLU)",
    tags=["korean", "comprehensive", "mcq"],
    model=model,
    benchmarks=[
        BenchmarkConfig(
            id="click",
            provider_id=KOREAN_PROVIDER_ID,
            parameters={"temperature": 0.0, "max_tokens": 1024, "limit": 20},
        ),
        BenchmarkConfig(
            id="kmmlu",
            provider_id=KOREAN_PROVIDER_ID,
            parameters={"temperature": 0.0, "max_tokens": 1024, "limit": 20},
        ),
    ],
    experiment=ExperimentConfig(
        name="korean-mcq-comprehensive-evaluation",
        tags=[
            ExperimentTag(key="evaluation_type", value="comprehensive"),
            ExperimentTag(key="model", value=MODEL_NAME),
            ExperimentTag(key="language", value="korean"),
            ExperimentTag(key="adapter", value="korean-mcq"),
        ],
    ),
)

print("Multi-Benchmark Job Request:")
print(f"  Name:       {multi_job_request.name}")
print(f"  Benchmarks: {[b.id for b in multi_job_request.benchmarks]}")
print(f"  Experiment: {multi_job_request.experiment.name}")

# Uncomment to submit:
# multi_job = client.jobs.submit(multi_job_request)
# print(f"\nJob submitted: {multi_job.id}")

### Step 11: Use Collections for Standardized Evaluations

Collections group benchmarks into reusable evaluation suites. This is useful for certification or compliance workflows.

In [ ]:
try:
    collections = client.collections.list()
    print(f"Available Collections ({len(collections)}):")
    print("=" * 60)
    for coll in collections:
        print(f"\n  Collection: {coll.name}")
        print(f"  ID:         {coll.resource.id}")
        print(f"  Category:   {coll.category}")
        print(f"  Benchmarks: {len(coll.benchmarks)}")
        for bm_ref in coll.benchmarks[:5]:
            print(f"    - {bm_ref.id} (provider: {bm_ref.provider_id})")
        if len(coll.benchmarks) > 5:
            print(f"    ... and {len(coll.benchmarks) - 5} more")
except Exception as e:
    print(f"Failed to list collections: {e}")

### Step 12: List and Manage Jobs

Review all submitted evaluation jobs.

In [ ]:
try:
    jobs_list = client.jobs.list()
    print(f"Evaluation Jobs ({len(jobs_list)}):")
    print("=" * 60)
    for j in jobs_list:
        state = j.effective_state.value
        exp_name = j.experiment.name if j.experiment else "N/A"
        bm_ids = [b.id for b in j.benchmarks] if j.benchmarks else []
        print(f"  [{state:>16s}] {j.id[:12]}... | {j.name} | exp={exp_name} | benchmarks={bm_ids}")
except Exception as e:
    print(f"Failed to list jobs: {e}")

## Reference: EvalHub SDK Quick Reference

### Client SDK Imports

```python
from evalhub import (
    SyncEvalHubClient,          # Synchronous client (recommended for notebooks)
    AsyncEvalHubClient,         # Async client (for production apps)
    ModelConfig,                # Model endpoint configuration
    BenchmarkConfig,            # Benchmark selection and parameters
    JobSubmissionRequest,       # Full job request
    ExperimentConfig,           # MLflow experiment settings
    ExperimentTag,              # MLflow tags
    JobStatus,                  # Job status enum
)
```

### Key API Patterns

```python
# Initialize client
client = SyncEvalHubClient(
    base_url="https://evalhub:8443",
    auth_token="...",           # Optional: SA token or API key
    insecure=True,              # Skip TLS verification (dev only)
    tenant="my-namespace",      # Kubernetes namespace
)

# Explore resources (all return plain lists)
providers: list[Provider]     = client.providers.list()
benchmarks: list[Benchmark]   = client.benchmarks.list()
collections: list[Collection] = client.collections.list()

# Submit a job
job: EvaluationJob = client.jobs.submit(request)

# Monitor and retrieve results
status: EvaluationJob = client.jobs.get(job.id)
```

### MLflow Experiment Structure

When an `ExperimentConfig` is provided:

- **Experiment Name**: `{prefix}_{experiment.name}`
- **Tags**: Direct mapping from `experiment.tags`
- **Run**: One MLflow run per evaluation request
- **Metrics**: Benchmark scores logged automatically
- **Parameters**: Model config and benchmark settings logged
- **Artifacts**: Detailed result files stored

### Useful Links

- [EvalHub GitHub](https://github.com/eval-hub/eval-hub)
- [EvalHub SDK GitHub](https://github.com/eval-hub/eval-hub-sdk)
- [EvalHub API Docs](https://eval-hub.github.io/eval-hub/)
- [MLflow Integration Guide](https://github.com/eval-hub/eval-hub/blob/main/MLFLOW.md)

## Done!

You've now configured the EvalHub SDK and learned how to:

1. **Connect** to the EvalHub service with the Python SDK
2. **Configure a model endpoint** pointing to your deployed InferenceService
3. **Set up MLflow experiment tracking** with tags and experiment names
4. **Submit evaluations** using lm-evaluation-harness benchmarks
5. **Monitor** job progress and **retrieve results**
6. **Run multi-benchmark** evaluations in a single request

### Next Steps

- **1_eval_hub_guidellm_benchmark/** -- Run inference performance benchmarks via GuideLLM
- **2_eval_hub_kmcq_benchmark/1_kmcq_benchmark.ipynb** -- Run Korean MCQ benchmark evaluation
- **2_eval_hub_kmcq_benchmark/2_summarize_results.ipynb** -- Summarize results and generate reports
- **3_eval_hub_unified_benchmark/1_unified_benchmark.ipynb** -- Run unified accuracy + performance evaluation